# The `cgns` driver — reading a CFD grid

**What the driver does.** **CGNS** (CFD General Notation System) is a standard, HDF5-based file format for
computational-fluid-dynamics meshes. The `cgns` driver reads the mesh (its surface nodes and faces) into a
pyramids `MeshDataset`.

**The sample file.** `examples/data/real-data/cgns/cgns_dpw4crm_..._orphan98.cgns` is a **263 MB** NASA
**Common Research Model (CRM)** grid — an **aircraft surface** together with the surrounding **simulation box**
(the outer boundary of the airflow domain). It is a **3-D** grid.

## 1. Read the file

## Getting the data

The sample file(s) this notebook reads are **not stored in the repository** (they are large and carry per-source licensing). The cell below downloads just this driver's data from the project's public **Google Drive** folder ([link](https://drive.google.com/drive/folders/16IkuKUkCo21uDb_X1iZciSPp7X99bmjv)) into `examples/data/real-data/cgns/` — but only if it is not already there, so it is a no-op once you have the data. It uses the standard library only.

In [ ]:
# --- Getting the data (download it if you do not already have it) ------------
# The real-data samples are not committed to the repository (large, per-source
# licensing). If the file this notebook needs is not already under
# examples/data/real-data/, download it from the project's public Google Drive
# folder below. Standard library only.
#   https://drive.google.com/drive/folders/16IkuKUkCo21uDb_X1iZciSPp7X99bmjv
from pathlib import Path
import urllib.request, urllib.parse, http.cookiejar, re, html, zipfile

DATA = Path("../../../examples/data/real-data")

def _download_drive(file_id, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    op = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))
    op.addheaders = [("User-Agent", "Mozilla/5.0")]
    base = "https://drive.usercontent.google.com/download"
    r = op.open(base + "?" + urllib.parse.urlencode({"id": file_id, "export": "download", "confirm": "t"}))
    if r.headers.get_content_type() == "text/html":          # virus-scan page -> re-submit its form
        form = dict(re.findall(r'name="([^"]+)"\s+value="([^"]*)"', r.read().decode("utf-8", "replace")))
        r = op.open(base + "?" + urllib.parse.urlencode(form))
    with open(dest, "wb") as fh:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)

marker = DATA / "cgns" / "cgns_dpw4crm_unstructured_mixed_g0-coarse_2.07mn_orphan98.cgns"
if not marker.exists():
    print("Downloading cgns sample (~264 MB) from Google Drive ...")
    _download_drive("1dBeDhS4g1pzyTm89SpVP5UUmWhDK2Aqx", marker)
    print("Saved:", marker)
else:
    print("cgns data already present:", marker)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from cleopatra.config import Config

from pyramids.netcdf.ugrid import MeshDataset

DATA = Path("../../../examples/data/real-data")
grid = DATA / "cgns" / "cgns_dpw4crm_unstructured_mixed_g0-coarse_2.07mn_orphan98.cgns"

ds = MeshDataset.read_file(grid, driver="cgns")
Config.set_matplotlib_backend()
type(ds).__name__

## 2. Explore the mesh — attributes

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(b, 2) for b in ds.bounds),
    "variable_names": ds.variable_names,
})

**50 316 nodes**, **58 480 faces**. `crs`/`epsg` are `None` — a CFD grid lives in the solver's own
Cartesian coordinates, not a map projection. There is one data field, `node_z` (the **height** of each node,
since the surface is 3-D).

## 3. How faces are built — the connectivity

`k = 4`, but the grid is **mixed**: some faces are quadrilaterals (4 valid nodes), some triangles (3 valid
nodes, the spare slot padded with `-1`).

In [ ]:
conn = ds.mesh.face_node_connectivity.data
widths = (conn >= 0).sum(axis=1)   # count the valid (non -1) nodes per face
print("connectivity shape:", conn.shape)
pd.Series({int(k): int((widths == k).sum()) for k in np.unique(widths)},
          name="faces").rename_axis("nodes/face")

About **42 000 quadrilaterals** and **16 000 triangles** — a mixed-element surface in one table.

## 4. The data field — `node_z`

`node_z` is the **height** of each node. We check its range, but we will **not colour-map it**: almost all the
nodes sit on the flat top of the simulation box at one height, which swamps the aircraft. So for this grid we
look at the **geometry** (the mesh edges) rather than a height colour-map.

In [ ]:
z = ds.masked_values("node_z")
print("node_z (height): min %.2f, max %.2f" % (z.min(), z.max()))

## 5. Plot the grid — `ds.plot_outline()`

`ds.plot_outline()` draws the **edges of every cell** (the mesh skeleton) through cleopatra and returns a
`MeshGlyph`. It shows what the grid is made of, without needing a data field.

In [ ]:
glyph = ds.plot_outline(linewidth=0.2)
glyph.ax.set_aspect("equal")
glyph.ax.set_title("The whole grid: aircraft + simulation box")

**What you are looking at:** the outline of every cell, seen from above. The big, coarse **rectangle of
cells** is the **simulation box** — the outer boundary of the airflow domain. The tiny dense blob at the bottom
centre is the **aircraft**. Most of a CFD grid is the empty space *around* the object, not the object itself.

## 6. Zoom to the aircraft — `crop()` + `plot_outline()`

`crop(bbox=...)` keeps only the faces inside a box, so we cut down to the aircraft and draw its surface mesh.

In [ ]:
aircraft = ds.crop(bbox=(0.0, 0.0, 1.1, 0.4))
print("faces in whole grid:", ds.n_face, " -> around the aircraft:", aircraft.n_face)

glyph = aircraft.plot_outline(linewidth=0.3)
glyph.ax.set_aspect("equal")
glyph.ax.set_title("Zoomed to the aircraft surface")

**What you are looking at:** the **edges of the aircraft's surface cells**, now large enough to see. The
long flat strip along `y = 0` is the **fuselage**; the raised block rising from it is the **wing** (only half
the aircraft is modelled, because it is left–right symmetric). The dense lines are the individual quad/tri cells
of the CFD surface mesh.

## Takeaway

- The `cgns` driver reads a large CFD container (**263 MB**) into a pyramids `MeshDataset` in about a second.
- Explore it with the usual attributes and methods (`face_node_connectivity`, `masked_values()`, `crop()`), and
  plot it with `ds.plot_outline()` / `ds.plot(...)` (both return a cleopatra `MeshGlyph`).
- The grid is **3-D** (an aircraft inside a simulation box); `plot_outline()` shows that structure, and `crop()`
  zooms down to the aircraft surface.